# SeaDronesSee ConvNeXt QAT - Kaggle 2xT4
Notebook này chạy theo kiểu **1 epoch mỗi lần chạy** để phù hợp với giới hạn Kaggle, nhưng tự resume từ checkpoint trên Kaggle Dataset.

- FP32 stage sẽ tự dùng **2 GPU T4** khi Kaggle có đủ 2 GPU, nhờ `train_fp32_ddp.py`.
- QAT stage dùng **2 GPU T4** qua `torch.distributed.run`.
- PT2E là nhánh CPU/x86 riêng trong repo, nên không đưa vào notebook GPU này.

Workflow: clone repo -> tải data -> restore checkpoint -> build runtime config -> chạy epoch tiếp theo -> upload checkpoint dataset -> benchmark khi huấn luyện xong.

In [ ]:
import os
import shutil
import subprocess
import sys
import datetime
import json
from pathlib import Path

import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

ROOT = Path("/kaggle/working")
REPO = ROOT / "EchteAI"
WORK = ROOT / "seadronessee_kaggle_run_fresh"
OUTPUT = WORK / "checkpoints"
LOGS = WORK / "logs"
OUTPUT.mkdir(parents=True, exist_ok=True)
LOGS.mkdir(parents=True, exist_ok=True)
FRESH_START = True

def run_and_log(command, log_path, cwd=None):
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("Command:", " ".join(map(str, command)), flush=True)
    print("Log:", log_path, flush=True)
    with log_path.open("a", encoding="utf-8") as log_file:
        log_file.write(f"\n===== START {datetime.datetime.now().isoformat()} =====\n")
        log_file.write(" ".join(map(str, command)) + "\n")
        log_file.flush()
        process = subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, env=env, cwd=cwd,
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line)
            log_file.flush()
        code = process.wait()
        log_file.write(f"===== END code={code} {datetime.datetime.now().isoformat()} =====\n")
    if code:
        raise subprocess.CalledProcessError(code, command)

def checkpoint_epoch(path):
    path = Path(path)
    if not path.is_file():
        return 0
    payload = torch.load(path, map_location="cpu", weights_only=False)
    return int(payload.get("epoch", 0)) if isinstance(payload, dict) else 0

def print_checkpoint_summary(folder):
    folder = Path(folder)
    for path in sorted(folder.glob("*")):
        if path.is_file():
            print(f"{path.name:30s} {path.stat().st_size / 2**20:9.2f} MB")


## 1. Clone repo và cài dependency
Notebook này cài thêm `kagglehub` để tải dataset/checkpoint và dùng `torch.distributed.run` cho QAT 2 GPU.

In [ ]:
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/NguyenDucThang-tb/EchteAI.git", str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[coco,pt2e]", "kagglehub"], check=True)
print("Repository:", Path.cwd())


## 2. Khai báo dataset và tải dữ liệu
Bạn chỉ cần giữ đúng `KAGGLE_USERNAME`. Notebook sẽ tải SeaDronesSee từ Kaggle Dataset đã có sẵn và khôi phục checkpoint phiên trước nếu tìm thấy dataset checkpoint.

In [ ]:
import kagglehub
import yaml

KAGGLE_USERNAME = "nguyenducthangtb"
SEADRONESSEE_DATASET = f"{KAGGLE_USERNAME}/seadronessee-compressed"
CHECKPOINT_DATASET = f"{KAGGLE_USERNAME}/echteai-seadronessee-m3-checkpoints"
VARIANT = "M3"
QAT_NUM_GPUS = 2

assert KAGGLE_USERNAME != "YOUR_USERNAME", "Hãy điền Kaggle username của bạn"
print("Data dataset:", SEADRONESSEE_DATASET)
print("Checkpoint dataset:", CHECKPOINT_DATASET)


In [ ]:
def find_dataset_root(start):
    start = Path(start)
    candidates = [start, *[p.parent.parent for p in start.rglob("instances_train.json")]]
    for candidate in candidates:
        if (candidate / "annotations/instances_train.json").exists() and (candidate / "images/train").is_dir():
            return candidate
    raise FileNotFoundError(f"Không tìm thấy cấu trúc images/ + annotations/ trong {start}")

DATA_DOWNLOAD = Path(kagglehub.dataset_download(SEADRONESSEE_DATASET))
DATA_ROOT = find_dataset_root(DATA_DOWNLOAD)
print("DATA_ROOT:", DATA_ROOT)

if FRESH_START:
    print("Fresh start enabled: skipping checkpoint restore and training from epoch 0.")
else:
    try:
        previous = Path(kagglehub.dataset_download(CHECKPOINT_DATASET))
        copied = 0
        for source in previous.rglob("*"):
            if source.is_file():
                target = OUTPUT / source.name
                shutil.copy2(source, target)
                copied += 1
        print(f"Restored {copied} files from {previous}")
    except Exception as error:
        print("No previous checkpoint dataset found; starting fresh.")
        print("Reason:", error)


## 3. Tạo runtime config cho Kaggle
Cấu hình này giữ nguyên pipeline hiện tại: anchor auto-fit và selective QAT. QAT batch size là batch trên mỗi GPU, nên `qat_batch_size=1` sẽ cho global batch 2 trên Kaggle 2xT4.

In [ ]:
base = yaml.safe_load(Path("configs/seadronessee_colab.yaml").read_text())
base["dataset"].update({
    "train_images": str(DATA_ROOT / "images/train"),
    "train_annotations": str(DATA_ROOT / "annotations/instances_train.json"),
    "val_images": str(DATA_ROOT / "images/val"),
    "val_annotations": str(DATA_ROOT / "annotations/instances_val.json"),
    "test_images": str(DATA_ROOT / "images/val"),
    "test_annotations": str(DATA_ROOT / "annotations/instances_val.json"),
})
base["training"].update({
    "fp32_batch_size": 2,
    "qat_batch_size": 1,
    "fp32_epochs": 30,
    "qat_epochs": 11,
    "epoch_benchmark_images": 100,
    "print_frequency": 50,
})
base["quantization"]["backend"] = "auto"
base["quantization"]["variant"] = VARIANT
base["output"] = {
    "directory": str(OUTPUT),
    "fp32_best": str(OUTPUT / "fp32_best.pt"),
    "fp32_last": str(OUTPUT / "fp32_last.pt"),
    "qat_best": str(OUTPUT / "qat_best.pt"),
    "qat_last": str(OUTPUT / "qat_last.pt"),
    "int8_model": str(OUTPUT / "selective_int8.pt"),
    "evaluation_json": str(OUTPUT / "evaluation.json"),
    "benchmark_json": str(OUTPUT / "benchmark.json"),
    "epoch_benchmarks": str(OUTPUT / "epoch_benchmarks.json"),
}
base["benchmark"] = {"warmup_iterations": 10, "iterations": 50, "num_threads": 1}
RUNTIME_CONFIG = WORK / "runtime.yaml"
RUNTIME_CONFIG.write_text(yaml.safe_dump(base, sort_keys=False), encoding="utf-8")
print("Runtime config:", RUNTIME_CONFIG)
print(RUNTIME_CONFIG.read_text())


## 4. Chạy epoch tiếp theo
Ô này tự quyết định stage tiếp theo dựa trên checkpoint hiện có:
- còn thiếu `fp32_last.pt` thì chạy FP32 tiếp một epoch
- khi FP32 xong thì chuyển sang QAT; nếu có 2 GPU thì dùng DDP để tận dụng cả hai
- mỗi lần chạy chỉ thực hiện 1 epoch để phù hợp với giới hạn runtime Kaggle

In [ ]:
fp32_last = OUTPUT / "fp32_last.pt"
qat_last = OUTPUT / "qat_last.pt"
fp32_total = int(base["training"]["fp32_epochs"])
qat_total = int(base["training"]["qat_epochs"])
fp32_done = checkpoint_epoch(fp32_last)
qat_done = checkpoint_epoch(qat_last)
print(f"FP32={fp32_done}/{fp32_total} | QAT={qat_done}/{qat_total}")

if fp32_done < fp32_total:
    next_stage = "fp32"
elif qat_done < qat_total:
    next_stage = "qat"
else:
    next_stage = "done"
print("Next stage:", next_stage)

if next_stage == "fp32":
    command = [
        sys.executable, "-u", "scripts/train_next_epoch.py",
        "--config", str(RUNTIME_CONFIG),
        "--stage", "fp32",
    ]
    if fp32_last.exists():
        command += ["--resume", str(fp32_last)]
    run_and_log(command, LOGS / "fp32_train.log", cwd=REPO)
elif next_stage == "qat":
    if torch.cuda.device_count() >= 2:
        command = [
            sys.executable, "-m", "torch.distributed.run", "--standalone", "--nproc_per_node=2",
            "scripts/train_qat_ddp.py",
            "--config", str(RUNTIME_CONFIG),
            "--fp32-checkpoint", str(OUTPUT / "fp32_best.pt"),
            "--variant", VARIANT,
            "--epochs-this-run", "1",
            "--no-find-unused-parameters",
        ]
        if qat_last.exists():
            command += ["--resume", str(qat_last)]
        run_and_log(command, LOGS / "qat_train_ddp.log", cwd=REPO)
    else:
        command = [
            sys.executable, "-u", "scripts/train_qat.py",
            "--config", str(RUNTIME_CONFIG),
            "--fp32-checkpoint", str(OUTPUT / "fp32_best.pt"),
            "--variant", VARIANT,
            "--epochs-this-run", "1",
        ]
        if qat_last.exists():
            command += ["--resume", str(qat_last)]
        run_and_log(command, LOGS / "qat_train.log", cwd=REPO)
else:
    print("Training already complete; chạy cell benchmark bên dưới.")

print("Current checkpoint summary:")
print_checkpoint_summary(OUTPUT)


## 5. Upload checkpoint dataset
Mỗi lần train xong một epoch, upload toàn bộ output folder lên Kaggle Dataset để phiên sau resume được ngay.

In [ ]:
fp32_epoch = checkpoint_epoch(OUTPUT / "fp32_last.pt")
qat_epoch = checkpoint_epoch(OUTPUT / "qat_last.pt")
version_notes = f"FP32 epoch {fp32_epoch}, QAT epoch {qat_epoch}"
print("Version notes:", version_notes)
print_checkpoint_summary(OUTPUT)
kagglehub.dataset_upload(CHECKPOINT_DATASET, str(OUTPUT), version_notes=version_notes)
print("Checkpoint dataset uploaded:", CHECKPOINT_DATASET)


## 6. Benchmark và visualization
Chạy sau khi FP32 xong và QAT xong. Cell benchmark so sánh FP32 với selective INT8 trên CPU. Cell visualization tạo ảnh GT / FP32 / INT8 để xem trực quan.

In [ ]:
fp32_total = int(base["training"]["fp32_epochs"])
qat_total = int(base["training"]["qat_epochs"])
fp32_done = checkpoint_epoch(OUTPUT / "fp32_last.pt")
qat_done = checkpoint_epoch(OUTPUT / "qat_last.pt")
int8_model = OUTPUT / "selective_int8.pt"
if fp32_done < fp32_total or qat_done < qat_total:
    print("Chưa train xong, bỏ qua benchmark.")
elif not int8_model.is_file():
    print("Chưa có selective_int8.pt, bỏ qua benchmark.")
else:
    benchmark_json = OUTPUT / "fp32_int8_comparison.json"
    benchmark_log = LOGS / "fp32_int8_comparison.log"
    command = [
        sys.executable, "-u", "scripts/benchmark.py",
        "--config", str(RUNTIME_CONFIG),
        "--fp32-checkpoint", str(OUTPUT / "fp32_best.pt"),
        "--int8-checkpoint", str(int8_model),
        "--output", str(benchmark_json),
    ]
    run_and_log(command, benchmark_log, cwd=REPO)
    print("Saved benchmark:", benchmark_json)
    print(benchmark_json.read_text())

    vis_dir = WORK / "visualizations"
    command = [
        sys.executable, "-u", "scripts/visualize_fp32_int8.py",
        "--config", str(RUNTIME_CONFIG),
        "--fp32-checkpoint", str(OUTPUT / "fp32_best.pt"),
        "--int8-checkpoint", str(int8_model),
        "--images", "100",
        "--score-threshold", "0.5",
        "--threads", "1",
        "--output-dir", str(vis_dir),
    ]
    run_and_log(command, LOGS / "visualize_fp32_int8.log", cwd=REPO)
    print("Visualization output:", vis_dir)
